In [ ]:
import os
from typing import TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY was not found. Add it to a .env file in the repository root."
    )


In [ ]:
 llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)


In [ ]:
class JokeState(TypedDict, total=False):
    topic: str
    joke: str
    explanation: str


In [ ]:
def generate_joke(state: JokeState):
    prompt = f"Generate a short, funny joke about the topic: {state['topic']}"
    response = llm.invoke(prompt).content
    return {"joke": response}


def generate_explanation(state: JokeState):
    prompt = f"Explain the humor in this joke in a simple way: {state['joke']}"
    response = llm.invoke(prompt).content
    return {"explanation": response}


In [ ]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)


In [ ]:
config1 = {"configurable": {"thread_id": "1"}}

result1 = workflow.invoke({"topic": "pizza"}, config=config1)

print("Topic:", result1["topic"])
print("Joke:", result1["joke"])
print("Explanation:", result1["explanation"])


In [ ]:
snapshot1 = workflow.get_state(config1)
snapshot1.values

list(workflow.get_state_history(config1))


In [ ]:
result2 = workflow.invoke({"topic": "space"}, config=config1)

print("Updated topic:", result2["topic"])
print("New joke:", result2["joke"])
print("New explanation:", result2["explanation"])

workflow.get_state(config1).values


In [ ]:
config2 = {"configurable": {"thread_id": "2"}}

result3 = workflow.invoke({"topic": "cricket"}, config=config2)
print("Thread 2 state:", workflow.get_state(config2).values)

print("\nThread 1 state:", workflow.get_state(config1).values)
